In [ ]:
# 避免 TensorFlow / protobuf 版本衝突
!pip install protobuf==4.25.3

In [ ]:
# 載入所有會用到的套件
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Dense, GlobalAveragePooling1D,
                                     LayerNormalization, Dropout, MultiHeadAttention,
                                     Embedding, concatenate)
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.metrics import F1Score

# 載入 dataset
DATA_PATH = '/kaggle/input/introduction-to-data-secience-ttmatch/'
train_df = pd.read_csv(f"{DATA_PATH}train.csv")
test_df = pd.read_csv(f"{DATA_PATH}test.csv")

# 檢查
print("Train DataFrame shape:", train_df.shape)
print("Test DataFrame shape:", test_df.shape)
train_df.head()

In [ ]:
# 特徵工程
def feature_engineering(df):
    # 計算得分差距
    df['scoreDiff'] = df['scoreSelf'] - df['scoreOther']
    # 判斷是否為發球方
    df['is_server'] = (df['gamePlayerId'] == df['serveId']).astype(int)
    return df
train_df = feature_engineering(train_df)
test_df = feature_engineering(test_df)

train_df[[
    'sex', 'strickNumber', 'scoreDiff', 'is_server','strickId', 'handId',
    'strengthId', 'spinId', 'pointId', 'actionId', 'positionId'
]].head()

In [ ]:
# 將資料分成離散欄位以及數值欄位
categorical_cols = [
    'sex', 'strickId', 'handId', 'strengthId', 'spinId', 
    'pointId', 'actionId', 'positionId'
]
numerical_cols = ['strickNumber', 'scoreDiff', 'is_server']

# 所有分類特徵的 LabelEncoder
feature_encoders = {}

for col in categorical_cols:
    if col in train_df.columns:
        # 合併訓練與測試資料的所有可能值
        all_possible_values = pd.concat([train_df[col], test_df[col]]).unique()
        
        # 用所有可能的值去 fit Encoder，得到 id → encoded id 的對映 
        encoder = LabelEncoder()
        encoder.fit(all_possible_values)
        feature_encoders[col] = encoder
        
        # 轉換訓練集與測試集的欄位
        train_df[col] = encoder.transform(train_df[col])
        test_df[col] = encoder.transform(test_df[col])

# 取得編碼後的類別總數，模型的輸出層會用到
n_action_classes = len(feature_encoders['actionId'].classes_)
n_point_classes  = len(feature_encoders['pointId'].classes_)

# 按照 rally_uid 順序切割訓練集和驗證集
unique_rallies = train_df['rally_uid'].unique()
n_train = int(len(unique_rallies) * 0.8)

# 前 80% 作為訓練集，後 20% 作為驗證集
train_rallies = unique_rallies[:n_train]
val_rallies = unique_rallies[n_train:]

# 檢查
print(f"訓練集 rallies 數量: {len(train_rallies)}")
print(f"驗證集 rallies 數量: {len(val_rallies)}")

# 初始化訓練集和驗證集的字典
X_train = {col: [] for col in categorical_cols}
X_train['numerical_input'] = []
y_train = {'action_output': [], 'point_output': [], 'rally_output': []}
X_val = {col: [] for col in categorical_cols}
X_val['numerical_input'] = []
y_val = {'action_output': [], 'point_output': [], 'rally_output': []}

# 設定序列最大長度
sequence_length = 64

# === 處理訓練集 ===
for rally_uid in train_rallies:
    rally_df = train_df[train_df['rally_uid'] == rally_uid].sort_values('strickNumber')
    
    if len(rally_df) < 2:
        continue
    
    # 只產生最後 K 個樣本
    K = 6 # 可調整
    start_idx = max(1, len(rally_df) - K)
    
    for i in range(start_idx, len(rally_df)):
        features_df = rally_df.iloc[:i]  # 用前 i 拍
        target_df = rally_df.iloc[i]     # 預測第 i+1 拍
        
        # 加入 X_train
        for col in categorical_cols:
            X_train[col].append(features_df[col].values)
        X_train['numerical_input'].append(features_df[numerical_cols].values)
        
        # 加入 y_train
        y_train['action_output'].append(target_df['actionId'])
        y_train['point_output'].append(target_df['pointId'])
        y_train['rally_output'].append(rally_df.iloc[-1]['serverGetPoint'])  # 使用最終結果
        
# === 處理驗證集 ===
for rally_uid in val_rallies:
    rally_sequence_df = train_df[train_df['rally_uid'] == rally_uid].sort_values('strickNumber')
    
    if len(rally_sequence_df) < 2:
        continue

    # 只產生最後 K 個樣本
    K = 6 # 可調整
    start_idx = max(1, len(rally_sequence_df) - K)
    
    for i in range(start_idx, len(rally_sequence_df)):
        features_df = rally_sequence_df.iloc[:i]  # 用前 i 拍
        target_df = rally_sequence_df.iloc[i]     # 預測第 i+1 拍

        # 加入 X_val
        for col in categorical_cols:
            X_val[col].append(features_df[col].values)
        X_val['numerical_input'].append(features_df[numerical_cols].values)

        # 加入 y_val
        y_val['action_output'].append(target_df['actionId'])
        y_val['point_output'].append(target_df['pointId'])
        y_val['rally_output'].append(rally_sequence_df.iloc[-1]['serverGetPoint'])  # 使用最終結果

# === Padding 訓練集 ===
for col in categorical_cols:
    X_train[col] = pad_sequences(X_train[col], maxlen=sequence_length, padding='pre', truncating='pre', value=0)
X_train['numerical_input'] = pad_sequences(X_train['numerical_input'], maxlen=sequence_length, padding='pre', truncating='pre', value=0, dtype='float32')

y_train['action_output'] = np.array(y_train['action_output'])
y_train['point_output'] = np.array(y_train['point_output'])
y_train['rally_output'] = np.array(y_train['rally_output'])

# === Padding 驗證集 ===
for col in categorical_cols:
    X_val[col] = pad_sequences(X_val[col], maxlen=sequence_length, padding='pre', truncating='pre', value=0)
X_val['numerical_input'] = pad_sequences(X_val['numerical_input'], maxlen=sequence_length, padding='pre', truncating='pre', value=0, dtype='float32')

y_val['action_output'] = np.array(y_val['action_output'])
y_val['point_output'] = np.array(y_val['point_output'])
y_val['rally_output'] = np.array(y_val['rally_output'])

# 檢查
print("資料準備完成！")
print("訓練集樣本數:", len(X_train['sex']))
print("驗證集樣本數:", len(X_val['sex']))

In [ ]:
# 模型參數
sequence_length = 64 # 與資料準備階段一致
embedding_dim = 32
num_heads = 4
ff_dim = 128

# --- 1. 為每個分類特徵建立 Input 和 Embedding 層 ---
input_layers = []
embedding_layers = []

for col in categorical_cols:
    encoder = feature_encoders[col]
    vocab_size = len(encoder.classes_)
    
    # 建立輸入層，名稱必須和 X 字典的 key 一致
    input_layer = Input(shape=(sequence_length,), name=col)
    input_layers.append(input_layer)
    
    # 建立 Embedding 層
    embedding_layer = Embedding(input_dim=vocab_size,
                                output_dim=embedding_dim, 
                                name=f'{col}_embedding')(input_layer)
    embedding_layers.append(embedding_layer)

# 為數值特徵建立 Input 層
# 數值特徵的維度是 sequence_length*len(numerical_cols)
numerical_input_layer = Input(shape=(sequence_length, len(numerical_cols)), name='numerical_input')
input_layers.append(numerical_input_layer)

# --- 2. 將所有 Embedding 向量在特徵維度上串接起來 ---
# 將分類特徵串接起來
concatenated_embeddings = concatenate(embedding_layers, name='concatenated_embeddings')
# 將數值特徵也串接進來
concatenated_features = concatenate([concatenated_embeddings, numerical_input_layer], name='concatenated_features')

# 使用一個 Dense 層來統一維度，確保可以被 num_heads 整除
total_dim = embedding_dim * len(categorical_cols)
x = Dense(total_dim, activation='relu', name='feature_projection')(concatenated_features)

# --- 3. 將串接後的向量輸入到 Transformer Block ---
attention_output = MultiHeadAttention(
    num_heads=num_heads,
    key_dim=total_dim // num_heads,
    name='multi_head_attention'
)(x, x)
attention_output = Dropout(0.1)(attention_output)
x1 = LayerNormalization(epsilon=1e-6)(x + attention_output)

# Feed Forward Block
ffn = Dense(ff_dim, activation="relu")(x1)
ffn = Dense(total_dim)(ffn) # 確保維度匹配
ffn = Dropout(0.1)(ffn)
x2 = LayerNormalization(epsilon=1e-6)(x1 + ffn)

# --- 4. 全域池化與輸出層 ---
pooled_output = GlobalAveragePooling1D()(x2)
pooled_output = Dropout(0.2)(pooled_output)

# 三個任務的輸出層
action_output = Dense(n_action_classes, activation='softmax', name='action_output')(pooled_output)
point_output = Dense(n_point_classes, activation='softmax', name='point_output')(pooled_output)
rally_output = Dense(1, activation='sigmoid', name='rally_output')(pooled_output)


# --- 5. 建立與編譯模型 ---
model = Model(inputs=input_layers, outputs=[action_output, point_output, rally_output])

# 自定義 F1 metric，因為 F1Score 這個 Keras metric 預期的是 one-hot label，而訓練時用的是 sparse label
_action_f1_obj = F1Score(average='macro', name='f1_action') 
def action_f1(y_true, y_pred):
    # y_true: (batch,) 整數標籤 -> one-hot: (batch, n_action_classes)
    y_true_oh = tf.one_hot(tf.cast(y_true, tf.int32), depth=n_action_classes)
    return _action_f1_obj(y_true_oh, y_pred)

_point_f1_obj = F1Score(average='macro', name='f1_point')
def point_f1(y_true, y_pred):
    y_true_oh = tf.one_hot(tf.cast(y_true, tf.int32), depth=n_point_classes)
    return _point_f1_obj(y_true_oh, y_pred)


# 編譯模型，loss 和 loss_weights 的 key 要對應輸出層的名稱
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
              loss={
                  'action_output': 'sparse_categorical_crossentropy',
                  'point_output': 'sparse_categorical_crossentropy',
                  'rally_output': 'binary_crossentropy'
              },
              loss_weights={'action_output': 0.4, 'point_output': 0.4, 'rally_output': 0.2},
              metrics={
                  'action_output': [action_f1], 
                  'point_output': [point_f1], 
                  'rally_output': tf.keras.metrics.AUC(name='auc')
              })

# 印出模型結構檢查
model.summary()


In [ ]:
# 進行訓練
history = model.fit(
    X_train, y_train,
    batch_size=64,
    epochs=70,
    validation_data=(X_val, y_val)
)

print("Transformer model training finished.")

In [ ]:
# --- 1. 準備測試資料 ---
# 初始化一個字典來存放測試集的特徵序列
X_test = {col: [] for col in categorical_cols}
X_test['numerical_input'] = []
test_rally_uids = test_df['rally_uid'].unique()

# 序列長度必須與訓練時一致
sequence_length = 64

# 遍歷測試集中的每個 rally_uid 來建立序列
for rally_uid in test_rally_uids:
    rally_sequence_df = test_df[test_df['rally_uid'] == rally_uid].sort_values('strickNumber')
    
    # 使用完整已知序列預測下一個未知拍
    features_df = rally_sequence_df
    for col in categorical_cols:
        X_test[col].append(features_df[col].values)
    X_test['numerical_input'].append(features_df[numerical_cols].values)

# 對每個特徵序列進行 padding
for col in categorical_cols:
    X_test[col] = pad_sequences(X_test[col], maxlen=sequence_length, padding='pre', truncating='pre', value=0)
X_test['numerical_input'] = pad_sequences(X_test['numerical_input'], maxlen=sequence_length, padding='pre', truncating='pre', value=0, dtype='float32')

print("測試資料準備完成！")

# --- 2. 進行預測 ---
action_preds, point_preds, rally_preds = model.predict(X_test)

# 模型的輸出是每個分類的機率分佈，要用 argmax 找到機率最高的類別索引
predicted_action_indices = np.argmax(action_preds, axis=1)
predicted_point_indices = np.argmax(point_preds, axis=1)

# --- 3. 建立並儲存提交檔案 ---
# 使用 Encoder 的 inverse_transform 將索引還原為原始ID
final_action_preds = feature_encoders['actionId'].inverse_transform(predicted_action_indices)
final_point_preds  = feature_encoders['pointId'].inverse_transform(predicted_point_indices)

# 載入提交檔案
submission_df = pd.read_csv('/kaggle/input/ttmatch-dataset/sample_submission.csv')

# 將還原後的 ID 填入 submission_df
submission_df['actionId'] = final_action_preds
submission_df['pointId'] = final_point_preds
submission_df['serverGetPoint'] = rally_preds

# 儲存提交的檔案
submission_df.to_csv('submission.csv', index=False)

In [ ]:
# 繪製 Loss 曲線
plt.figure(figsize=(20, 5))

# 子圖1: Total Loss
plt.subplot(1, 4, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Total Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# 子圖2: Action Output Loss
plt.subplot(1, 4, 2)
plt.plot(history.history['action_output_loss'], label='Train Loss')
plt.plot(history.history['val_action_output_loss'], label='Val Loss')
plt.title('Action Output Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# 子圖3: Point Output Loss
plt.subplot(1, 4, 3)
plt.plot(history.history['point_output_loss'], label='Train Loss')
plt.plot(history.history['val_point_output_loss'], label='Val Loss')
plt.title('Point Output Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# 子圖4: Rally Output Loss
plt.subplot(1, 4, 4)
plt.plot(history.history['rally_output_loss'], label='Train Loss')
plt.plot(history.history['val_rally_output_loss'], label='Val Loss')
plt.title('Rally Output Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# 繪製 Metrics 曲線
plt.figure(figsize=(15, 5))

# 子圖1: Action F1
plt.subplot(1, 3, 1)
plt.plot(history.history['action_output_action_f1'], label='Train F1')
plt.plot(history.history['val_action_output_action_f1'], label='Val F1')
plt.title('Action Output F1')
plt.xlabel('Epoch')
plt.ylabel('F1 score')
plt.legend()
plt.grid(True)

# 子圖2: Point F1
plt.subplot(1, 3, 2)
plt.plot(history.history['point_output_point_f1'], label='Train F1')
plt.plot(history.history['val_point_output_point_f1'], label='Val F1')
plt.title('Point Output F1')
plt.xlabel('Epoch')
plt.ylabel('F1 score')
plt.legend()
plt.grid(True)

# 子圖3: Rally AUC
plt.subplot(1, 3, 3)
plt.plot(history.history['rally_output_auc'], label='Train AUC')
plt.plot(history.history['val_rally_output_auc'], label='Val AUC')
plt.title('Rally Output AUC')
plt.xlabel('Epoch')
plt.ylabel('AUC')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()
